In [3]:
import numpy as np
import pandas as pd

In [4]:
sales = pd.read_csv('sales.csv')
inventories = pd.read_csv('inventories.csv')
satisfaction = pd.read_csv('satisfaction.csv')

df_sales = pd.DataFrame(sales)
df_inventories = pd.DataFrame(inventories)
df_satisfaction = pd.DataFrame(satisfaction)

In [5]:
# Limpieza de datos (eliminación de NaN)
df_sales.dropna()
df_inventories.dropna()
df_satisfaction.dropna()

,ID_Tienda,Satisfacción_Promedio,Fecha_Evaluación
0,1,85,2023-01-15
1,2,90,2023-01-15
2,3,70,2023-01-15
3,4,65,2023-01-15
4,5,55,2023-01-15


## Exploración de datos

In [6]:
df_sales.dtypes

ID_Tienda           int64
Producto              str
Cantidad_Vendida    int64
Precio_Unitario     int64
Fecha_Venta           str
dtype: object

In [7]:
# Ventas totales por producto y tienda
df_sales.groupby(['Producto', 'ID_Tienda'])['Cantidad_Vendida'].sum()

Producto    ID_Tienda
Producto A  1            20
            2            30
            3            10
            4            25
Producto B  1            15
            3            40
            5            20
Producto C  2            25
            4            35
            5            30
Name: Cantidad_Vendida, dtype: int64

In [11]:
# Ingresos totales por tienda
df_sales['Importe_Total'] = df_sales['Cantidad_Vendida'] * df_sales['Precio_Unitario']
df_sales.groupby('ID_Tienda')['Importe_Total'].sum()

ID_Tienda
1     5000
2    10500
3     9000
4    13000
5    13000
Name: Importe_Total, dtype: int64

In [12]:
# Resumen estadístico de ventas
df_sales.describe()

,ID_Tienda,Cantidad_Vendida,Precio_Unitario,Importe_Total
count,10.000000,10.000000,10.000000,10.000000
mean,3.000000,25.000000,190.000000,5050.000000
std,1.490712,9.128709,87.559504,3361.960407
min,1.000000,10.000000,100.000000,1000.000000
25%,2.000000,20.000000,100.000000,2625.000000
50%,3.000000,25.000000,200.000000,3500.000000
75%,4.000000,30.000000,275.000000,7875.000000
max,5.000000,40.000000,300.000000,10500.000000


In [32]:
# Promedio de ventas por tienda
df_sales.groupby('ID_Tienda')['Importe_Total'].mean()
# Promedio de ventas por tienda y categoría de producto
df_sales.groupby(['ID_Tienda', 'Producto'])['Importe_Total'].mean()

ID_Tienda  Producto  
1          Producto A     2000.0
           Producto B     3000.0
2          Producto A     3000.0
           Producto C     7500.0
3          Producto A     1000.0
           Producto B     8000.0
4          Producto A     2500.0
           Producto C    10500.0
5          Producto B     4000.0
           Producto C     9000.0
Name: Importe_Total, dtype: float64

## Análisis de inventarios

In [14]:
df_inventories.dtypes

ID_Tienda              int64
Producto                 str
Stock_Disponible       int64
Fecha_Actualización      str
dtype: object

In [16]:
# Cálculo de rotación de inventario por tienda
# (división ventas totales por stock disponible por producto)
ventas_tienda = df_sales.groupby('ID_Tienda')['Cantidad_Vendida'].sum()
rotacion_inventario = df_inventories.merge(ventas_tienda, on='ID_Tienda').rename(columns={'Cantidad_Vendida': 'Ventas_Tienda'})
rotacion_inventario['Rotacion'] = rotacion_inventario['Ventas_Tienda'] / rotacion_inventario['Stock_Disponible']
rotacion_inventario = rotacion_inventario.drop(columns='Ventas_Tienda')
print(rotacion_inventario)

   ID_Tienda    Producto  Stock_Disponible Fecha_Actualización  Rotacion
0          1  Producto A                50          2023-01-05  0.700000
1          1  Producto B                40          2023-01-06  0.875000
2          2  Producto A                60          2023-01-07  0.916667
3          2  Producto C                45          2023-01-08  1.222222
4          3  Producto A                30          2023-01-09  1.666667
5          3  Producto B                80          2023-01-10  0.625000
6          4  Producto C                70          2023-01-11  0.857143
7          4  Producto A                50          2023-01-12  1.200000
8          5  Producto B                40          2023-01-13  1.250000
9          5  Producto C                60          2023-01-14  0.833333


In [17]:
# Almacena resultado en nueva columna de DataFrame Inventario
df_inventories = df_inventories.merge(rotacion_inventario[['ID_Tienda', 'Producto', 'Rotacion']], on=['ID_Tienda', 'Producto'])
print(df_inventories)

   ID_Tienda    Producto  Stock_Disponible Fecha_Actualización  Rotacion
0          1  Producto A                50          2023-01-05  0.700000
1          1  Producto B                40          2023-01-06  0.875000
2          2  Producto A                60          2023-01-07  0.916667
3          2  Producto C                45          2023-01-08  1.222222
4          3  Producto A                30          2023-01-09  1.666667
5          3  Producto B                80          2023-01-10  0.625000
6          4  Producto C                70          2023-01-11  0.857143
7          4  Producto A                50          2023-01-12  1.200000
8          5  Producto B                40          2023-01-13  1.250000
9          5  Producto C                60          2023-01-14  0.833333


In [18]:
# Tiendas con niveles críticos de inventario. Aquellas que tengan % productos vendidos < 10% del stock disponible
ventas_stock = df_sales[['ID_Tienda', 'Producto', 'Cantidad_Vendida']].merge(df_inventories[['ID_Tienda', 'Producto', 'Stock_Disponible']], on=['ID_Tienda', 'Producto'])
ventas_stock['Porcentaje_Vendido'] = ventas_stock['Cantidad_Vendida'] / ventas_stock['Stock_Disponible'] * 100
stock_limit = 10
tiendas_criticas = ventas_stock.query('Porcentaje_Vendido < @stock_limit')[['ID_Tienda', 'Porcentaje_Vendido']]
print(tiendas_criticas)

Empty DataFrame
Columns: [ID_Tienda, Porcentaje_Vendido]
Index: []


## Satisfacción del cliente

In [40]:
# Análisis de la satisfacción de los clientes en cada tienda
# Relación con el rendimiento de las ventas
rendimiento_ventas = (df_sales.groupby('ID_Tienda')['Importe_Total']
                      .agg(['sum', 'mean'])
                      .reset_index()
                      .rename(columns={
                            'sum': 'Ventas_Totales',
                            'mean': 'Ventas_Promedio'
                        })
)
relacion_satisfaccion_rendimiento = pd.merge(
    df_satisfaction[['ID_Tienda', 'Satisfacción_Promedio']],
    rendimiento_ventas,
    on='ID_Tienda',
    how='inner'
)
print(relacion_satisfaccion_rendimiento)

   ID_Tienda  Satisfacción_Promedio  Ventas_Totales  Ventas_Promedio
0          1                     85            5000           2500.0
1          2                     90           10500           5250.0
2          3                     70            9000           4500.0
3          4                     65           13000           6500.0
4          5                     55           13000           6500.0


Se puede ver no hay una relación clara entre la satisfacción del cliente en cada tienda y la cantidad de ventas de la misma.

In [26]:
# Filtrado de tiendas con niveles bajos de satisfacción (<60%)
satisfaccion_limit = 60
tiendas_baja_satisfaccion = df_satisfaction.query('Satisfacción_Promedio < @satisfaccion_limit')
print(tiendas_baja_satisfaccion)

   ID_Tienda  Satisfacción_Promedio Fecha_Evaluación
4          5                     55       2023-01-15


### Recomendaciones:
Se recomienda realize un estudio más exhaustivo en esta tienda para ver por qué los clientes no salen tan satisfechos como en las otras sucursales.
Se trata de falta de personal o de trato hacia los clientes?

## Operaciones con Numpy

In [28]:
# Cálculo mediana y desviación ventas totales
ventas_totales = df_sales['Importe_Total'].to_numpy()
print(ventas_totales)
mediana = np.median(ventas_totales)
std = np.std(ventas_totales)
media = np.mean(ventas_totales)
print(mediana, std, media)

[ 2000  3000  3000  7500  1000  8000 10500  2500  4000  9000]
3500.0 3189.4356867634124 5050.0


In [29]:
# Estimación de ventas
rng = np.random.seed(42)
narray = 90
estimacion_ventas_totales = np.random.normal(mediana, std, narray)
cantidades = df_sales['Cantidad_Vendida'].to_numpy()
estimacion_cantidades = np.random.normal(cantidades.mean(), cantidades.std(), narray)
estimacion_precios_unitarios = estimacion_ventas_totales / estimacion_cantidades
tiendas = np.random.choice(range(1, 6), narray)
productos = np.random.choice(['Producto A', 'Producto B', 'Producto C'], narray)
estimacion_ventas = pd.DataFrame({
    'ID_Tienda': tiendas,
    'Producto': productos,
    'Cantidad_Vendida': estimacion_cantidades,
    'Precio_Unitario': estimacion_precios_unitarios,
    'Importe_Total': estimacion_ventas_totales
})
print(estimacion_ventas)

    ID_Tienda    Producto  Cantidad_Vendida  Precio_Unitario  Importe_Total
0           5  Producto C         25.840716       196.752977    5084.237846
1           4  Producto C         33.388712        91.618237    3059.014904
2           5  Producto B         18.920042       294.172760    5565.760937
3           3  Producto C         22.162363       377.108070    8357.605776
4           4  Producto B         21.604244       127.437132    2753.182870
..        ...         ...               ...              ...            ...
85          2  Producto C         32.163617        59.062952    1899.678179
86          4  Producto A         25.112600       255.633278    6419.616182
87          2  Producto A         37.587974       121.010259    4548.530521
88          3  Producto C         22.708005        79.723601    1810.363901
89          1  Producto B         48.557356       105.793105    5137.033468

[90 rows x 5 columns]
